# B29 Corridor — MLP Regressor für Kraftstoffpreise

**Strecke:** Aalen → Schwäbisch Gmünd → Schorndorf → Stuttgart  
**Ziel:** Stündliche Kraftstoffpreise je Cluster vorhersagen (48h-Horizont) um den optimalen Tankzeitpunkt auf der B29 zu identifizieren.

Vorgehen nach CRISP-DM (6 Phasen).

In [ ]:
import sys, os
sys.path.insert(0, '..')   # project root → makes 'scripts' importable

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor

from scripts.data_transform import B29DataLoader, B29_CLUSTERS_DEFAULT

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
SEED = 42
print('Imports OK')

---
## §1 Business Understanding

Ein Logistikbetrieb fährt täglich die B29 zwischen Aalen und Stuttgart.  
Ziel: Vorhersage der stündlichen Kraftstoffpreise je Streckenabschnitt für die nächsten 48 Stunden — so kann der günstigste Tankzeitpunkt je Cluster geplant werden.

In [ ]:
# Fleet economics on the B29
TRUCKS          = 5
KM_PER_DAY      = 200          # km/day per truck (Aalen ↔ Stuttgart round trip)
CONSUMPTION     = 30           # L / 100 km
DAILY_LITERS    = TRUCKS * KM_PER_DAY * CONSUMPTION / 100

PRICE_SWING_TYPICAL = 0.05     # 5 ct/L typical intraday swing

print(f'Daily diesel need   : {DAILY_LITERS:,.0f} L')
print(f'Cost at 1.70 €/L    : {DAILY_LITERS * 1.70:,.2f} €')
print(f'Saving if 5 ct cheaper: {DAILY_LITERS * PRICE_SWING_TYPICAL:,.2f} € / day')
print(f'Annual saving potential: {DAILY_LITERS * PRICE_SWING_TYPICAL * 250:,.0f} € (250 working days)')

---
## §2 Data Understanding

Tankerkönig liefert event-basierte Preisänderungen (~327k/Tag, gesamt 87 GB).  
Die vier B29-Cluster werden nach PLZ gefiltert und stündlich gemittelt (Event-Mean-Ansatz).

In [ ]:
# Show cluster definitions
print('B29 Cluster-Definitionen (PLZ-Codes):')
for name, plzs in B29_CLUSTERS_DEFAULT.items():
    print(f'  {name:22s}: {plzs}')

In [ ]:
# Load data (uses cache if available, otherwise scans raw CSVs)
loader = B29DataLoader(stride=0, forecast_horizon=24, fuel_type='diesel', cache=True)
X, y = loader.load()

print(f'\nZeitraum : {y.index[0]}  →  {y.index[-1]}')
print(f'Stunden  : {len(y):,}')
print(f'Cluster  : {list(y.columns)}')

In [ ]:
# Price summary per cluster
y.describe().round(4)

In [ ]:
# Time series overview — all clusters
fig, ax = plt.subplots(figsize=(14, 4))
for col in y.columns:
    ax.plot(y.index, y[col], lw=0.4, alpha=0.7, label=col.replace('diesel_', ''))
ax.set(title='Stündlicher Dieselpreis je B29-Cluster (2014–2026)',
       ylabel='€/L', xlabel='')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Intraday pattern (hour-of-day median per cluster)
fig, ax = plt.subplots(figsize=(10, 4))
for col in y.columns:
    intraday = y[col].groupby(y.index.hour).median()
    ax.plot(intraday.index, intraday.values, marker='o', ms=3,
            label=col.replace('diesel_', ''))
ax.set(title='Intraday-Muster: Median-Dieselpreis je Stunde',
       xlabel='Stunde', ylabel='€/L')
ax.set_xticks(range(0, 24, 2))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cluster correlation
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(y.corr(), annot=True, fmt='.3f', cmap='Blues', ax=ax,
            xticklabels=[c.replace('diesel_', '') for c in y.columns],
            yticklabels=[c.replace('diesel_', '') for c in y.columns])
ax.set_title('Korrelation Dieselpreis zwischen Clustern')
plt.tight_layout()
plt.show()

---
## §3 Data Preparation

Der `B29DataLoader` hat bereits alle Features berechnet (Lags, Rolling, Trend, Momentum, Zeitmerkmale).  
Hier prüfen wir die Feature-Matrix und führen den Temporal-Split durch.

In [ ]:
print(f'Feature-Matrix X: {X.shape}')
print(f'Target-Matrix  y: {y.shape}')
print(f'\nFeature-Gruppen:')

lag_cols     = [c for c in X.columns if '_lag_' in c]
roll_cols    = [c for c in X.columns if '_roll_' in c]
diff_cols    = [c for c in X.columns if '_diff' in c or '_momentum' in c]
trend_cols   = [c for c in X.columns if '_trend' in c]
time_cols    = ['hour', 'day_of_week', 'is_weekend', 'is_holiday']

print(f'  Lag-Features        : {len(lag_cols):3d} Spalten')
print(f'  Rolling-Features    : {len(roll_cols):3d} Spalten')
print(f'  Diff/Momentum       : {len(diff_cols):3d} Spalten')
print(f'  Trend               : {len(trend_cols):3d} Spalten')
print(f'  Zeitmerkmale        : {len(time_cols):3d} Spalten')
print(f'  ──────────────────────────────')
print(f'  Total               : {X.shape[1]:3d} Spalten')

In [ ]:
# Check for remaining NaN after warmup drop
nan_count = X.isna().sum().sum()
print(f'NaN in X: {nan_count}  ({nan_count/X.size*100:.2f}%)')
if nan_count > 0:
    print('Columns with NaN:')
    print(X.isna().sum()[X.isna().sum() > 0])

In [ ]:
# Drop any remaining NaN rows (should be zero or very few)
valid_idx = X.dropna().index.intersection(y.dropna().index)
X = X.loc[valid_idx]
y = y.loc[valid_idx]

# Temporal split
X_train, X_val, X_test, y_train, y_val, y_test = loader.train_val_test_split(X, y)

print(f'\nFeature-Spalten (erste 10):')
for col in X.columns[:10]:
    print(f'  {col}')
print('  …')

In [ ]:
# Scale features (fit only on train set to avoid leakage)
scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_val_s   = scaler_X.transform(X_val)
X_test_s  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_s = scaler_y.fit_transform(y_train)
y_val_s   = scaler_y.transform(y_val)
# y_test bleibt unscaled für finale Evaluation

print('Scaler fitted on train set only — kein Data Leakage.')

---
## §4 Modeling

In [ ]:
# Baseline: DummyRegressor (mean prediction)
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train_s, y_train_s)

y_val_dummy = scaler_y.inverse_transform(dummy.predict(X_val_s))
dummy_mae = mean_absolute_error(y_val, y_val_dummy, multioutput='uniform_average')
dummy_r2  = r2_score(y_val, y_val_dummy, multioutput='uniform_average')
print(f'Baseline (DummyRegressor mean):')
print(f'  Val MAE = {dummy_mae:.4f} €/L')
print(f'  Val R²  = {dummy_r2:.4f}')

In [ ]:
# Cross-validation with TimeSeriesSplit (5 folds)
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=SEED,
    verbose=False,
)
# MultiOutputRegressor trains one MLP per cluster
model_cv = MultiOutputRegressor(mlp, n_jobs=-1)

tscv = TimeSeriesSplit(n_splits=5)
cv_scores = cross_val_score(
    model_cv, X_train_s, y_train_s,
    cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1
)
cv_mae = -cv_scores
print(f'TimeSeriesSplit CV (5 Folds) — MAE auf skalierten Targets:')
print(f'  Folds : {np.round(cv_mae, 5)}')
print(f'  Mean  : {cv_mae.mean():.5f} ± {cv_mae.std():.5f}')

In [ ]:
# Train final model on full train set
model = MultiOutputRegressor(
    MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=SEED,
    ),
    n_jobs=-1,
)
model.fit(X_train_s, y_train_s)

# Validation performance
y_val_pred = scaler_y.inverse_transform(model.predict(X_val_s))
val_mae = mean_absolute_error(y_val, y_val_pred, multioutput='raw_values')
val_r2  = r2_score(y_val, y_val_pred, multioutput='raw_values')

print('Validation Performance (2022–2023):')
print(f'{"Cluster":<22}  {"MAE (€/L)":>10}  {"R²":>8}')
print('-' * 44)
for col, mae, r2 in zip(y.columns, val_mae, val_r2):
    print(f'{col.replace("diesel_", ""):<22}  {mae:>10.5f}  {r2:>8.4f}')
print('-' * 44)
print(f'{"Gesamt (Ø)":<22}  {val_mae.mean():>10.5f}  {val_r2.mean():>8.4f}')

---
## §5 Evaluation (Test-Set — einmalig geöffnet)

In [ ]:
# Test set evaluation (2024-01 → present)
y_test_pred = scaler_y.inverse_transform(model.predict(X_test_s))
y_test_pred = pd.DataFrame(y_test_pred, index=y_test.index, columns=y_test.columns)

test_mae  = mean_absolute_error(y_test, y_test_pred, multioutput='raw_values')
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred, multioutput='raw_values'))
test_r2   = r2_score(y_test, y_test_pred, multioutput='raw_values')

print('Test Performance (2024–2026):')
print(f'{"Cluster":<22}  {"MAE":>8}  {"RMSE":>8}  {"R²":>8}')
print('-' * 52)
for col, mae, rmse, r2 in zip(y.columns, test_mae, test_rmse, test_r2):
    print(f'{col.replace("diesel_", ""):<22}  {mae:>8.5f}  {rmse:>8.5f}  {r2:>8.4f}')
print('-' * 52)
print(f'{"Gesamt (Ø)":<22}  {test_mae.mean():>8.5f}  {test_rmse.mean():>8.5f}  {test_r2.mean():>8.4f}')

improvement = (dummy_mae - test_mae.mean()) / dummy_mae * 100
print(f'\nVerbesserung ggü. Baseline: {improvement:.1f}%')
print(f'Fleet-Kostenfehler/Tag    : {test_mae.mean() * DAILY_LITERS:.2f} €')

In [ ]:
# Visual: Actual vs. Predicted for a 2-week window per cluster
sample_start = y_test.index[0]
sample_end   = y_test.index[0] + pd.Timedelta(days=14)

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
for ax, col in zip(axes.flat, y_test.columns):
    label = col.replace('diesel_', '')
    mask = (y_test.index >= sample_start) & (y_test.index <= sample_end)
    ax.plot(y_test.index[mask], y_test.loc[mask, col], lw=1.2,
            label='Actual', color='steelblue')
    ax.plot(y_test_pred.index[mask], y_test_pred.loc[mask, col],
            lw=1, ls='--', label='Predicted', color='tomato', alpha=0.8)
    ax.set_title(label)
    ax.set_ylabel('€/L')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m'))
    ax.tick_params(axis='x', rotation=30)
fig.suptitle('Actual vs. Predicted — 14-Tage-Fenster (Test-Set)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution per cluster
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, col in zip(axes, residuals.columns):
    ax.hist(residuals[col].dropna(), bins=60, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(0, color='red', lw=1)
    ax.set_title(col.replace('diesel_', ''))
    ax.set_xlabel('Residual (€/L)')
fig.suptitle('Residualverteilung je Cluster')
plt.tight_layout()
plt.show()

---
## §6 Deployment: 48h-Forecast je B29-Cluster

Rekursive Vorhersage der nächsten 48 Stunden.  
Empfehlung: **Wo** und **wann** auf der B29 tanken?

In [ ]:
def forecast_48h(
    model,
    scaler_X: StandardScaler,
    scaler_y: StandardScaler,
    X: pd.DataFrame,
    y: pd.DataFrame,
    horizon: int = 48,
) -> pd.DataFrame:
    """
    Recursive 48h forecast.
    At each step we take the last known feature row, predict the next
    price, then manually update the lag columns for the following step.
    """
    price_cols = list(y.columns)
    lag_map = {1: [], 2: [], 3: [], 6: [], 12: [], 24: [], 48: [], 72: [], 168: []}
    # Identify lag columns for each cluster
    lag_col_map = {}
    for col in price_cols:
        lag_col_map[col] = {lag: f'{col}_lag_{lag}h' for lag in [1, 2, 3, 6, 12, 24, 48, 72, 168]}

    feature_row = X.iloc[[-1]].copy()   # last known row
    history     = y.iloc[-200:].copy()  # recent price history for lag lookup

    forecasts = []
    last_ts   = y.index[-1]

    for step in range(1, horizon + 1):
        ts = last_ts + pd.Timedelta(hours=step)

        # Update time features
        feature_row['hour']        = ts.hour
        feature_row['day_of_week'] = ts.dayofweek
        feature_row['is_weekend']  = int(ts.dayofweek >= 5)

        # Predict
        x_scaled = scaler_X.transform(feature_row)
        y_scaled = model.predict(x_scaled)
        y_pred   = scaler_y.inverse_transform(y_scaled)[0]

        pred_row = dict(zip(price_cols, y_pred))
        pred_row['timestamp'] = ts
        forecasts.append(pred_row)

        # Append prediction to history for next step's lag features
        new_y_row = pd.DataFrame([y_pred], index=[ts], columns=price_cols)
        history   = pd.concat([history, new_y_row])

        # Update lag columns in feature_row
        for col in price_cols:
            for lag in [1, 2, 3, 6, 12, 24, 48, 72, 168]:
                lag_col = lag_col_map[col][lag]
                if lag_col in feature_row.columns:
                    lag_ts = ts - pd.Timedelta(hours=lag)
                    if lag_ts in history.index:
                        feature_row[lag_col] = history.loc[lag_ts, col]

    df_fc = pd.DataFrame(forecasts).set_index('timestamp')
    return df_fc


df_forecast = forecast_48h(model, scaler_X, scaler_y, X_test, y_test)
print(f'Forecast shape: {df_forecast.shape}')
df_forecast.head()

In [ ]:
# Plot 48h forecast per cluster
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['steelblue', 'darkorange', 'green', 'red']

for col, color in zip(df_forecast.columns, colors):
    label = col.replace('diesel_', '')
    ax.plot(df_forecast.index, df_forecast[col], lw=1.5, label=label, color=color)
    ax.fill_between(df_forecast.index, df_forecast[col] - 0.005,
                    df_forecast[col] + 0.005, alpha=0.15, color=color)

ax.set(title='48h-Diesel-Preisvorhersage je B29-Cluster',
       ylabel='€/L', xlabel='')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m %H:%M'))
ax.tick_params(axis='x', rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Recommendation: optimal refuel cluster & time
min_price_per_cluster = df_forecast.min()
min_time_per_cluster  = df_forecast.idxmin()
best_cluster = min_price_per_cluster.idxmin().replace('diesel_', '')
best_time    = min_time_per_cluster[min_price_per_cluster.idxmin()]
best_price   = min_price_per_cluster.min()
current_price = df_forecast.iloc[0].mean()

print('=== Tankempfehlung für die nächsten 48 Stunden ===')
print()
for col in df_forecast.columns:
    cluster = col.replace('diesel_', '')
    p_min = df_forecast[col].min()
    t_min = df_forecast[col].idxmin()
    print(f'  {cluster:22s}: {p_min:.4f} €/L  um {t_min.strftime("%d.%m.%Y %H:%M")}')

print()
print(f'>>> Empfehlung: {best_cluster} um {best_time.strftime("%d.%m.%Y %H:%M")} Uhr')
print(f'    Günstigster Preis : {best_price:.4f} €/L')
print(f'    Jetziger Schnitt  : {current_price:.4f} €/L')
saving = (current_price - best_price) * DAILY_LITERS
print(f'    Einsparung/Tag    : {saving:.2f} € ({current_price - best_price:.4f} €/L × {DAILY_LITERS:.0f} L)')